In [1]:
# ===========================================================================
# Trivariate token-level model: dist_ac, dist_art and duration as joint
# responses, instead of one long-format `dist` with a modality factor.
#
# Why this shape:
#   * one row per token, so nothing is duplicated across modality rows
#   * `modality` stops being a factor: its effects become DIFFERENCES between
#     response-specific coefficients, tested with hypothesis(). No effect
#     coding, no emmeans, no df puzzles.
#   * (1 | token_id) is replaced by the residual correlation, estimated
#     rather than assumed
#   * rescor(distac, distart) is a direct token-level measure of acoustic-
#     articulatory coupling -- low correlation IS motor equivalence
#
# Note on duration: with set_rescor(TRUE) a response cannot also appear on
# another response's right-hand side, so duration is NOT a covariate here.
# That is a feature: the surprisal coefficients on distac/distart are TOTAL
# effects, and duration's own equation carries the lengthening story
# separately. If you want the duration-adjusted (direct) effect instead, use
# the path model in section 6.
# ===========================================================================

library(dplyr)
library(tidyr)
library(brms)

options(mc.cores = parallel::detectCores())

LONG_PATH <- "../../../data_processing/combine_data/results_beus.csv"     # your current long-format frame
DUR_VAR   <- "phone2_duration"   # or: mutate(bigram_duration = phone1_duration + phone2_duration)

long <- read.csv(LONG_PATH)

long$phone1_duration <- as.numeric(long$phone1_end - long$phone1_start)
long$phone2_duration <- as.numeric(long$phone2_end - long$phone2_start)
long$bigram <- paste(long$phone1, long$phone2, sep = "_")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Rcpp

Warning message:
“package ‘Rcpp’ was built under R version 4.4.3”
Loading 'brms' package (version 2.23.0). Useful instructions
can be found by typing help('brms'). A more detailed introduction
to the package is available through vignette('brms_overview').


Attaching package: ‘brms’


The following object is masked from ‘package:stats’:

    ar




In [4]:
# ---------------------------------------------------------------------------
# 0. FIRST: is duration already inside the latent?
#
# You encode duration directly in z, so `dist` may contain duration
# mechanically. If it does, rescor(dist*, dur) is an encoder artifact and
# must not be read as a finding. Check before fitting anything.
# ---------------------------------------------------------------------------

long %>%
  group_by(modality) %>%
  summarise(
    r_dur2      = cor(dist, .data[[DUR_VAR]], use = "complete.obs"),
    r_dur1      = cor(dist, phone1_duration, use = "complete.obs"),
    r_absdiff   = cor(dist, abs(phone1_duration - phone2_duration),
                      use = "complete.obs"),
    r_sum       = cor(dist, phone1_duration + phone2_duration,
                      use = "complete.obs"),
    .groups = "drop"
  ) %>%
  print()

# Read it like this:
#   r_sum >> r_absdiff  -> a MAGNITUDE artifact: longer segment, bigger
#                          trajectory, bigger latent, bigger distances.
#                          Residualise dist on duration, or refit the
#                          autoencoder without duration in z, before
#                          interpreting rescor with dur.
#   r_absdiff >> r_sum  -> distance tracks duration MISMATCH, which is a
#                          real phonetic quantity and much less worrying.
#
# In python, on the vectors themselves:
#   np.corrcoef(np.linalg.norm(art, axis=1), durations)[0, 1]

# ---------------------------------------------------------------------------
# 1. Long -> wide, with integrity checks
# ---------------------------------------------------------------------------

TOKEN_LEVEL <- c("word", "bigram", "subject", "phone1", "phone2",
                 "bigram_surprisal", "phone1_duration", "phone2_duration",
                 "phone2_freq", "phone1_freq", "phone1_flexibility", "phone2_flexibility")

dat <- long %>%
  filter(!cross_syllable_boundary, data_space == "latent") %>%
  select(token_id, modality, dist, all_of(TOKEN_LEVEL))

# every token-level column must be constant within token_id, or pivot_wider
# silently produces list-columns
bad <- dat %>%
  group_by(token_id) %>%
  summarise(across(all_of(TOKEN_LEVEL), ~ n_distinct(.x) > 1), .groups = "drop") %>%
  select(-token_id) %>%
  summarise(across(everything(), sum))
stopifnot(all(bad == 0))

# exactly one ac row and one art row per token
counts <- dat %>% count(token_id, modality) %>% filter(n > 1)
if (nrow(counts) > 0) {
  stop(sprintf("%d token_id x modality cells have >1 row. If you merged two ",
               nrow(counts)),
       "annotation versions, the acoustic rows are duplicated -- de-duplicate ",
       "first, or give the two articulatory versions distinct modality labels ",
       "and add a fourth response.")
}

wide <- dat %>%
  pivot_wider(names_from = modality, values_from = dist, names_prefix = "dist")

stopifnot(all(c("distac", "distart") %in% names(wide)))

n_before <- nrow(wide)
wide <- wide %>% filter(!is.na(distac), !is.na(distart))
message(sprintf("tokens: %d -> %d after dropping incomplete pairs (%.1f%% lost)",
                n_before, nrow(wide), 100 * (1 - nrow(wide) / n_before)))

# A tibble: 2 × 5
  modality r_dur2 r_dur1 r_absdiff r_sum
  <chr>     <dbl>  <dbl>     <dbl> <dbl>
1 ac        0.274  0.332    0.127  0.393
2 art       0.139  0.125    0.0722 0.173


tokens: 139111 -> 135342 after dropping incomplete pairs (2.7% lost)



In [5]:
# ---------------------------------------------------------------------------
# 2. Scaling
#
# z-score each response so the ac-vs-art coefficient CONTRASTS are meaningful
# and rescor is interpretable. Centre the predictors so lower-order terms are
# read at the mean rather than at zero.
# ---------------------------------------------------------------------------

z <- function(x) as.numeric(scale(x))

# Drop bad rows BEFORE scaling, so the columns really are mean 0 / sd 1 in the
# data that gets fitted. log() silently produced NaNs in an earlier version of
# this pipeline and cost ~17k rows without warning -- fail loudly instead.
n_pre <- nrow(wide)
wide <- wide %>%
  filter(
    is.finite(.data[[DUR_VAR]]), .data[[DUR_VAR]] > 0,
    is.finite(phone2_freq),      phone2_freq > 0,
    is.finite(bigram_surprisal),
    is.finite(phone1_flexibility), is.finite(phone2_flexibility)
  )
if (nrow(wide) < n_pre)
  warning(sprintf(paste("dropped %d rows with non-positive or non-finite",
                        "duration/frequency/flexibility"), n_pre - nrow(wide)))

# by-subject scaling of duration variables
wide <- wide %>%
  group_by(subject) %>%
  mutate(
    dur1 = as.numeric(scale(log(phone1_duration))),
    dur2 = as.numeric(scale(log(phone2_duration))),
    dur  = as.numeric(scale(log(.data[[DUR_VAR]])))
  ) %>%
  ungroup()

wide <- wide %>%
  mutate(
    distac  = z(distac),
    distart = z(distart),
    ldur1    = z(dur1),
    ldur2    = z(dur2),
    surp    = z(bigram_surprisal),
    flex2   = z(phone2_flexibility),
    flex1   = z(phone1_flexibility),
    freq2   = z(log(phone2_freq)),
    freq1   = z(log(phone1_freq))
  )


stopifnot(all(sapply(wide[c("distac", "distart", "ldur1", "ldur2",
                            "surp", "flex2", "flex1", "freq2", "freq1")],
                     function(x) all(is.finite(x)) &&
                                 abs(mean(x)) < 1e-8 && abs(sd(x) - 1) < 1e-8)))

Warning message:
“dropped 40156 rows with non-positive or non-finite duration/frequency/flexibility”


In [6]:
library(dplyr)
library(brms)

options(mc.cores = parallel::detectCores())

# Set TRUE if section 4 says surprisal and duration interact. The simple a*b
# product is only valid without that interaction.
MEDIATOR_INTERACTION <- FALSE

# ---------------------------------------------------------------------------
# 1. Model
# ---------------------------------------------------------------------------

RE <- paste(
  "(1 |w| word)",
  "(1 + surp |b| bigram)",
  "(1 |p1| phone1)",
  "(1 |p2| phone2)",
  "(1 + surp |s| subject)",
  sep = " + "
)

PRED <- "surp * flex2 * flex1 + freq2 + freq1"

# add speech rate / prosodic position here -- see section 5
# PRED <- paste(PRED, "+ rate + final")

med <- if (MEDIATOR_INTERACTION) "surp * dur1 + surp * dur2" else "dur1 + dur2"

form <-
  bf(as.formula(paste("dur1    ~", PRED, "+", RE))) +
  bf(as.formula(paste("dur2    ~", PRED, "+", RE))) +
  bf(as.formula(paste("distac  ~", PRED, "+", med, "+", RE))) +
  bf(as.formula(paste("distart ~", PRED, "+", med, "+", RE))) +
  set_rescor(FALSE)

# In a multivariate model EVERY b / Intercept / sd / sigma is indexed by
# response (b_distac_surp, sd_word__distac_Intercept, ...), so a prior with no
# `resp` matches no parameter and brms rejects the whole set. Only `cor` (and
# `rescor`) are response-free, because the group-level correlations span
# responses -- that is what the |w| |b| |p1| |p2| |s| tags create.

RESP <- c("dur1", "dur2", "distac", "distart")

priors <- c(
  do.call(c, lapply(RESP, function(r) c(
    set_prior("normal(0, 1)",   class = "b",         resp = r),
    set_prior("normal(0, 2)",   class = "Intercept", resp = r),
    set_prior("exponential(1)", class = "sd",        resp = r),
    set_prior("exponential(1)", class = "sigma",     resp = r)
  ))),
  set_prior("lkj(2)", class = "cor")
)

# Fail here in seconds rather than after Stan compiles. If it errors, print
# default_prior(form, data = wide) and read off the exact class/resp/group
# combinations this model actually has. (Older brms: get_prior().)
validate_prior(priors, form, data = wide)

prior,class,coef,group,resp,dpar,nlpar,lb,ub,tag,source
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
"normal(0, 1)",b,,,distac,,,,,,user
,b,dur1,,distac,,,,,,default
,b,dur2,,distac,,,,,,default
,b,flex1,,distac,,,,,,default
,b,flex2,,distac,,,,,,default
,b,flex2:flex1,,distac,,,,,,default
,b,freq1,,distac,,,,,,default
,b,freq2,,distac,,,,,,default
,b,surp,,distac,,,,,,default


In [ ]:
fit1 <- brm(
  form, data = wide, prior = priors,
  chains = 1, iter = 4000,
  control = list(adapt_delta = 0.95, max_treedepth = 12),
  backend = "cmdstanr", threads = threading(2), seed = 1,
  file = "fit/fit_beus_chain1"
)
fit2 <- brm(
  form, data = wide, prior = priors,
  chains = 1, iter = 4000,
  control = list(adapt_delta = 0.95, max_treedepth = 12),
  backend = "cmdstanr", threads = threading(2), seed = 2,
  file = "fit/fit_beus_chain2"
)
fit3 <- brm(
  form, data = wide, prior = priors,
  chains = 1, iter = 4000,
  control = list(adapt_delta = 0.95, max_treedepth = 12),
  backend = "cmdstanr", threads = threading(2), seed = 3,
  file = "fit/fit_beus_chain3"
)
fit4 <- brm(
  form, data = wide, prior = priors,
  chains = 1, iter = 4000,
  control = list(adapt_delta = 0.95, max_treedepth = 12),
  backend = "cmdstanr", threads = threading(2), seed = 4,
  file = "fit/fit_beus_chain4"
)
fit <- combine_models(
  fit1,
  fit2,
  fit3,
  fit4
)
summary(fit)

Start sampling



In [ ]:
# ===========================================================================
# Inspecting the fitted path model.
#
#   PRED = surp * flex2 * flex1 + freq2 + freq1
#   mediators: dur1, dur2
#
# The surprisal slope is now a surface over two moderators:
#
#   slope(f1, f2) = b_surp + b_surp:flex2 * f2 + b_surp:flex1 * f1
#                          + b_surp:flex2:flex1 * f1 * f2
#
# so "flex2's effect", "flex1's effect" and "flex1:flex2's effect" are the
# slope along f2, the slope along f1, and the twist of that surface. Section E
# shows all three in one grid; section F shows the ac-art gap over the whole
# (f1, f2) plane.
#
# Order matters: convergence BEFORE effects.
# Assumes `fit` from path_model.R and `wide` from trivariate.R.
# ===========================================================================

library(brms); library(posterior); library(bayesplot)
library(ggplot2); library(dplyr)

RESP <- c("dur1", "dur2", "distac", "distart")

# ac is always blue, art always orange -- in the interval plots as a
# categorical pair (CVD dE 24.7 protan), and in the section-F heatmap as the
# two poles of a diverging scale, where the sign literally MEANS "ac larger"
# vs "art larger". Colour follows the entity in both roles. Neutral gray
# midpoint; never a hue at the midpoint.
PAL     <- c(ac = "#2a78d6", art = "#eb6834", shared = "#52514e")
MID     <- "#f0efec"
INK     <- "#0b0b0b"; INK2 <- "#52514e"
SURFACE <- "#fcfcfb"; GRID <- "#e6e5e1"; ZERO <- "#b8b7b2"

theme_paths <- function() {
  theme_minimal(base_size = 11) +
    theme(panel.background = element_rect(fill = SURFACE, colour = NA),
          plot.background  = element_rect(fill = SURFACE, colour = NA),
          panel.grid.minor = element_blank(),
          panel.grid.major.y = element_blank(),
          panel.grid.major.x = element_line(colour = GRID, linewidth = 0.3),
          axis.text = element_text(colour = INK2),
          axis.title = element_text(colour = INK2),
          plot.title = element_text(colour = INK, face = "bold"),
          plot.subtitle = element_text(colour = INK2),
          strip.text = element_text(colour = INK, face = "bold"),
          legend.position = "top", legend.title = element_blank(),
          legend.text = element_text(colour = INK2))
}

# ---------------------------------------------------------------------------
# A. Convergence gate
# ---------------------------------------------------------------------------

np   <- nuts_params(fit)
ndiv <- sum(subset(np, Parameter == "divergent__")$Value)
cat("divergent transitions :", ndiv, "\n")
cat("treedepth saturations :",
    sum(subset(np, Parameter == "treedepth__")$Value >= 12), "\n")

rh <- rhat(fit);       bad_rh <- sort(rh[rh > 1.01], decreasing = TRUE)
ne <- neff_ratio(fit); bad_ne <- sort(ne[ne < 0.1])
if (length(bad_rh)) { cat("\nRhat > 1.01:\n");   print(round(head(bad_rh, 15), 4)) }
if (length(bad_ne)) { cat("\nESS ratio < 0.1:\n"); print(round(head(bad_ne, 15), 3)) }
if (ndiv > 0 || length(bad_rh))
  cat("\n!! Do not read the paths yet. With a 4-equation model the subject term\n",
      "   is the usual culprit: raise adapt_delta to 0.99, then simplify it.\n\n")

# ---------------------------------------------------------------------------
# B. Coefficient accessor and the slope surface
#
# R orders interaction terms by formula expansion, so the three-way may be
# named surp:flex2:flex1 or surp:flex1:flex2 depending on how you wrote PRED.
# Match on the SET of terms instead of a literal string, and fail loudly.
# ---------------------------------------------------------------------------

d <- as_draws_df(fit)

cf <- function(resp, terms) {
  want <- sort(terms)
  pre  <- paste0("b_", resp, "_")
  cand <- grep(paste0("^", pre), names(d), value = TRUE)
  hit  <- cand[vapply(cand, function(nm)
    identical(sort(strsplit(sub(pre, "", nm, fixed = TRUE), ":")[[1]]), want),
    logical(1))]
  if (length(hit) != 1)
    stop(sprintf("expected 1 draw column for %s [%s], found %d%s",
                 resp, paste(want, collapse = ":"), length(hit),
                 if (length(hit)) paste0(": ", paste(hit, collapse = ", ")) else ""))
  d[[hit]]
}

# surprisal slope of `resp` at (f1, f2)
slope <- function(resp, f1, f2)
  cf(resp, "surp") +
  cf(resp, c("surp", "flex2")) * f2 +
  cf(resp, c("surp", "flex1")) * f1 +
  cf(resp, c("surp", "flex2", "flex1")) * f1 * f2

# b paths do not depend on flexibility (no dur x flex terms in PRED)
b1 <- list(ac = cf("distac", "dur1"), art = cf("distart", "dur1"))
b2 <- list(ac = cf("distac", "dur2"), art = cf("distart", "dur2"))

# the surprisal effect on distance, by route, at (f1, f2)
surp_effect <- function(path, mod, f1, f2) {
  r  <- if (mod == "ac") "distac" else "distart"
  i1 <- slope("dur1", f1, f2) * b1[[mod]]
  i2 <- slope("dur2", f1, f2) * b2[[mod]]
  switch(path,
         direct     = slope(r, f1, f2),
         `via dur1` = i1,
         `via dur2` = i2,
         indirect   = i1 + i2,
         total      = slope(r, f1, f2) + i1 + i2,
         stop("unknown path: ", path))
}

q  <- function(v, p) unname(quantile(v, p))
pd <- function(v) max(mean(v > 0), mean(v < 0))

# ---------------------------------------------------------------------------
# C. Path table and intervals, at the centre of the moderator space (0, 0)
# ---------------------------------------------------------------------------

row_of <- function(effect, modality, v)
  data.frame(effect = effect, modality = modality, mean = mean(v),
             lo95 = q(v, .025), lo50 = q(v, .25), hi50 = q(v, .75),
             hi95 = q(v, .975), pd = pd(v), stringsAsFactors = FALSE)

tab <- rbind(
  row_of("a1: surp to dur1",  "shared", slope("dur1", 0, 0)),
  row_of("a2: surp to dur2",  "shared", slope("dur2", 0, 0)),
  row_of("b1: dur1 to dist",  "ac",  b1$ac),  row_of("b1: dur1 to dist",  "art", b1$art),
  row_of("b2: dur2 to dist",  "ac",  b2$ac),  row_of("b2: dur2 to dist",  "art", b2$art)
)
for (p in c("direct", "via dur1", "via dur2", "indirect", "total"))
  for (m in c("ac", "art"))
    tab <- rbind(tab, row_of(p, m, surp_effect(p, m, 0, 0)))

print(within(tab, {
  mean <- round(mean,4); lo95 <- round(lo95,4); lo50 <- round(lo50,4)
  hi50 <- round(hi50,4); hi95 <- round(hi95,4); pd <- round(pd,3) }), row.names = FALSE)

plot_df <- tab
plot_df$row <- paste0(plot_df$effect,
                      ifelse(plot_df$modality == "shared", "",
                             paste0("  ", plot_df$modality)))
plot_df$row <- factor(plot_df$row, levels = rev(unique(plot_df$row)))

p_paths <- ggplot(plot_df, aes(y = row, x = mean, colour = modality)) +
  geom_vline(xintercept = 0, colour = ZERO, linewidth = 0.4) +
  geom_linerange(aes(xmin = lo95, xmax = hi95), linewidth = 0.6) +
  geom_linerange(aes(xmin = lo50, xmax = hi50), linewidth = 2) +
  geom_point(size = 3.6, colour = SURFACE) + geom_point(size = 2.6) +
  scale_colour_manual(values = PAL, breaks = c("ac", "art")) +
  labs(title = "Path effects", x = "effect (SD of each response)", y = NULL,
       subtitle = "at flex1 = flex2 = 0; thick 50% / thin 95% interval") +
  theme_paths()
print(p_paths)

# ---------------------------------------------------------------------------
# D. ac minus art -- FACETED, because b contrasts are duration slopes while
#    everything else is a surprisal slope. One shared x-axis would imply they
#    are comparable quantities; they are not.
# ---------------------------------------------------------------------------

crow <- function(contrast, kind, v)
  data.frame(contrast = contrast, kind = kind, mean = mean(v),
             lo95 = q(v, .025), lo50 = q(v, .25), hi50 = q(v, .75),
             hi95 = q(v, .975), pd = pd(v), stringsAsFactors = FALSE)

ctab <- rbind(
  crow("b1 (dur1 to dist)", "slope of DURATION",  b1$ac - b1$art),
  crow("b2 (dur2 to dist)", "slope of DURATION",  b2$ac - b2$art)
)
for (p in c("direct", "via dur1", "via dur2", "indirect", "total"))
  ctab <- rbind(ctab, crow(p, "slope of SURPRISAL",
                           surp_effect(p, "ac", 0, 0) - surp_effect(p, "art", 0, 0)))

print(within(ctab, {
  mean <- round(mean,4); lo95 <- round(lo95,4); lo50 <- round(lo50,4)
  hi50 <- round(hi50,4); hi95 <- round(hi95,4); pd <- round(pd,3) }), row.names = FALSE)

ctab$contrast <- factor(ctab$contrast, levels = rev(unique(ctab$contrast)))
p_contr <- ggplot(ctab, aes(y = contrast, x = mean)) +
  geom_vline(xintercept = 0, colour = ZERO, linewidth = 0.4) +
  geom_linerange(aes(xmin = lo95, xmax = hi95), linewidth = 0.6, colour = INK2) +
  geom_linerange(aes(xmin = lo50, xmax = hi50), linewidth = 2, colour = INK2) +
  geom_point(size = 3.6, colour = SURFACE) + geom_point(size = 2.6, colour = INK2) +
  facet_wrap(~ kind, scales = "free", ncol = 1) +
  labs(title = "ac minus art", x = "difference", y = NULL,
       subtitle = "negative = larger in the articulatory modality") +
  theme_paths()
print(p_contr)

# ---------------------------------------------------------------------------
# E. The three-way, as a grid
#
#   along x within a panel  = flex2's moderation of the surprisal slope
#   across columns          = flex1's moderation
#   change of SHAPE across columns = the flex1:flex2 interaction
#
# If the three lines in a row are parallel, there is no three-way: flex1 only
# shifts the slope. If they fan or cross, flex1 changes how flex2 matters.
# ---------------------------------------------------------------------------

PATHS  <- c("direct", "indirect", "total")   # or add "via dur1", "via dur2"
F1LEV  <- c(-1, 0, 1)
f2grid <- seq(-2, 2, length.out = 61)

band <- function(f1, f2, path, mod) {
  v <- surp_effect(path, mod, f1, f2)
  data.frame(f1 = f1, f2 = f2, path = path, modality = mod,
             mean = mean(v), lo = q(v, .025), hi = q(v, .975))
}
cond <- do.call(rbind, lapply(PATHS, function(p)
  do.call(rbind, lapply(F1LEV, function(f1)
    do.call(rbind, lapply(c("ac", "art"), function(m)
      do.call(rbind, lapply(f2grid, function(f2) band(f1, f2, p, m)))))))))

cond$path <- factor(cond$path, levels = PATHS)
cond$f1lab <- factor(sprintf("flex1 = %+d", cond$f1),
                     levels = sprintf("flex1 = %+d", F1LEV))

p_grid <- ggplot(cond, aes(f2, mean, colour = modality, fill = modality)) +
  geom_hline(yintercept = 0, colour = ZERO, linewidth = 0.4) +
  geom_ribbon(aes(ymin = lo, ymax = hi), alpha = 0.16, colour = NA) +
  geom_line(linewidth = 0.8) +
  geom_text(data = subset(cond, f2 == max(f2) & f1 == max(F1LEV)),
            aes(label = modality), hjust = -0.2, size = 3, show.legend = FALSE) +
  scale_colour_manual(values = PAL) + scale_fill_manual(values = PAL) +
  scale_x_continuous(expand = expansion(mult = c(0.02, 0.14))) +
  facet_grid(path ~ f1lab, scales = "free_y") +
  labs(title = "Surprisal slope across both flexibility moderators",
       subtitle = "mean and 95% interval; parallel lines across columns = no three-way",
       x = "phone2 flexibility (z)", y = "effect on distance") +
  theme_paths() +
  theme(panel.grid.major.y = element_line(colour = GRID, linewidth = 0.3))
print(p_grid)

# the table that ships with it
tbl <- subset(cond, f2 %in% c(-1, 0, 1))
tbl[, c("mean", "lo", "hi")] <- round(tbl[, c("mean", "lo", "hi")], 4)
print(tbl[order(tbl$path, tbl$modality, tbl$f1, tbl$f2),
          c("path", "modality", "f1", "f2", "mean", "lo", "hi")], row.names = FALSE)

# ---------------------------------------------------------------------------
# F. The ac-art gap over the whole (flex1, flex2) plane
#
# One signed scalar over two continuous dimensions -- this is the job a
# heatmap is for. Diverging scale: blue = larger in acoustics, orange = larger
# in articulation, neutral gray at zero. Solid contour marks the sign change;
# dashed contour marks where the posterior is 95% sure of the sign, which is
# the uncertainty a heatmap would otherwise throw away.
# ---------------------------------------------------------------------------

g <- expand.grid(f1 = seq(-2, 2, length.out = 31),
                 f2 = seq(-2, 2, length.out = 31),
                 path = PATHS, stringsAsFactors = FALSE)
gap <- t(mapply(function(f1, f2, p) {
  v <- surp_effect(p, "ac", f1, f2) - surp_effect(p, "art", f1, f2)
  c(mean = mean(v), pd = pd(v))
}, g$f1, g$f2, g$path))
g <- cbind(g, gap)
g$path <- factor(g$path, levels = PATHS)

lim <- max(abs(g$mean))
p_surf <- ggplot(g, aes(f1, f2)) +
  geom_raster(aes(fill = mean), interpolate = TRUE) +
  geom_contour(aes(z = mean), breaks = 0, colour = INK, linewidth = 0.5) +
  geom_contour(aes(z = pd), breaks = 0.95, colour = INK,
               linewidth = 0.4, linetype = "22") +
  scale_fill_gradient2(low = PAL[["art"]], mid = MID, high = PAL[["ac"]],
                       midpoint = 0, limits = c(-lim, lim),
                       name = "ac - art") +
  coord_fixed(expand = FALSE) +
  facet_wrap(~ path) +
  labs(title = "Where the modality gap lives",
       subtitle = paste("solid = zero crossing; dashed = 95% sure of the sign;",
                        "blue = larger in acoustics"),
       x = "phone1 flexibility (z)", y = "phone2 flexibility (z)") +
  theme_paths() +
  theme(legend.position = "right", legend.title = element_text(colour = INK2),
        panel.grid = element_blank())
print(p_surf)

# ---------------------------------------------------------------------------
# G. Mapping property or speaker property?
# ---------------------------------------------------------------------------

cr <- VarCorr(fit)$subject$cor
keep <- grep("surp", dimnames(cr)[[1]], value = TRUE)
print(round(cr[keep, "Estimate", keep], 3))

# ---------------------------------------------------------------------------
# H. Predictive checks
# ---------------------------------------------------------------------------

for (r in RESP) {
  print(pp_check(fit, resp = r, ndraws = 100) +
          labs(title = paste("posterior predictive:", r)) + theme_paths())
  cat(r, "bayes_R2:\n"); print(round(bayes_R2(fit, resp = r), 4))
}

# ggsave("fig_paths.png",   p_paths, width = 6.5, height = 5.5, dpi = 300, bg = SURFACE)
# ggsave("fig_contrast.png",p_contr, width = 6.0, height = 5.0, dpi = 300, bg = SURFACE)
# ggsave("fig_grid.png",    p_grid,  width = 8.5, height = 6.0, dpi = 300, bg = SURFACE)
# ggsave("fig_surface.png", p_surf,  width = 9.0, height = 3.6, dpi = 300, bg = SURFACE)